In [3]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

# Simulate demand data for a single e-bike parking spot over 60 days
np.random.seed(42)
T = 60  # time periods
num_features = 3  # e.g., temp, humidity, hour

# Generate mock features
temperature = 25 + 5 * np.sin(np.linspace(0, 3 * np.pi, T)) + np.random.normal(0, 1, T)
humidity = 60 + 10 * np.cos(np.linspace(0, 2 * np.pi, T)) + np.random.normal(0, 3, T)
hour = np.random.choice([8, 12, 18], size=T)  # simulate morning, midday, evening
X = np.column_stack([temperature, humidity, hour])
X = (X - X.mean(axis=0)) / X.std(axis=0)  # standardize

# Simulate regime changes at t=20 and t=40
true_states = np.array([0]*20 + [1]*20 + [2]*20)
betas = np.array([[1.0, 0.2, 0.3], [0.5, 0.5, 0.1], [1.5, -0.1, 0.2]])
y = np.array([X[t] @ betas[true_states[t]] + np.random.normal(0, 0.5) for t in range(T)])

# Model: switching linear regression with fixed 3 regimes (simplified)
with pm.Model() as model:
    # Hidden state indicator
    s = pm.Categorical('s', p=np.ones(3)/3, shape=T)

    # Regression weights for each regime
    beta = pm.Normal('beta', mu=0, sigma=1, shape=(3, num_features))
    sigma = pm.Exponential('sigma', 1.0)

    # Expected demand
    mu = pm.math.sum(X * beta[s], axis=1)

    # Likelihood
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)

    trace = pm.sample(1000, tune=1000, target_accept=0.9, chains=2, return_inferencedata=True)

# Summarize and visualize
az_summary = az.summary(trace, var_names=["beta", "sigma"])
tools.display_dataframe_to_user(name="Bayesian Model Summary", dataframe=az_summary)


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
Multiprocess sampling (2 chains in 2 jobs)
CompoundStep
>CategoricalGibbsMetropolis: [s]
>NUTS: [beta, sigma]


c:\ProgramData\anaconda3\envs\ra-route_network\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')